In [1]:
import hashlib
#HASH FUNCTION
def calculate_md5(input_string):
    md5_hash = hashlib.md5()
    md5_hash.update(input_string.encode('utf-8'))
    return md5_hash.hexdigest()

def make_plain_text(original_text):
    a = calculate_md5(original_text)
    return original_text+a

def build_key_map(key):
    sorted_key = sorted(list(key))
    return [sorted_key.index(k) for k in key]

#ENCRYPTION
def encrypt_transposition(plaintext, key):
    n = len(key)
    key_map = build_key_map(key)
    extra = len(plaintext) % n
    if extra != 0:
        plaintext += '$' * (n - extra)  
    rows = [plaintext[i:i+n] for i in range(0, len(plaintext), n)]
    # print(rows)
    ciphertext = []
    for col_index in range(n):
        actual_col = key_map.index(col_index)
        for row in rows:
            ciphertext.append(row[actual_col])
    return ''.join(ciphertext)

#DECRYPTION
def decrypt_transposition(ciphertext, key):
    n = len(key)
    key_map = build_key_map(key)
    num_rows = len(ciphertext) // n

    cols = [''] * n
    idx = 0
    for col_index in range(n):
        actual_col = key_map.index(col_index)
        cols[actual_col] = ciphertext[idx : idx + num_rows]
        idx += num_rows

    plaintext_chars = []
    for r in range(num_rows):
        for c in range(n):
            plaintext_chars.append(cols[c][r])
    plaintext = ''.join(plaintext_chars).rstrip('$')
    return plaintext

def pi_property(decrypted_text: str) -> bool:

  original_text = decrypted_text[:-32]
  hash_value = decrypted_text[-32:]

  return (hash_value == str(calculate_md5(original_text)))

In [2]:
key = input("Enter the key upto length 9 (Unique characters only): ")
def test_encrypt_decrypt(key):
    plaintexts = [
        "helloalgorithm",
        "securetransmission",
        "plaintextmessage",
        "cryptographyrules",
        "federatedlearning"
    ]
    for plaintext in plaintexts:
        print("Original text: ", plaintext)
        plaintext = make_plain_text(plaintext)
        print("Plain_text: ", plaintext)
        encrypted_text = encrypt_transposition(plaintext, key)
        print("Ciphertext: ",encrypted_text)
        decrypted_text = decrypt_transposition(encrypted_text,key)
        print("Decryptedtext: ",decrypted_text)
        print("Matches original plaintext?: ", pi_property(decrypted_text))
        print()

test_encrypt_decrypt(key)


Original text:  helloalgorithm
Plain_text:  helloalgorithmbf166fa043b9fd1b4b3ed9271db67922
Ciphertext:  lom631e19$elt10fb26$haifa949b2orbfbbdd2$lgh64d377$
Decryptedtext:  helloalgorithmbf166fa043b9fd1b4b3ed9271db67922
Matches original plaintext?:  True

Original text:  securetransmission
Plain_text:  securetransmission23b2023154ac651c86b42282c0e2605d
Ciphertext:  uas22ac205etmo255b26sesib16682rns33c82edcrin0414c0
Decryptedtext:  securetransmission23b2023154ac651c86b42282c0e2605d
Matches original plaintext?:  True

Original text:  plaintextmessage
Plain_text:  plaintextmessage5d46f828bd89a39deb827ea4e31338c8
Ciphertext:  ita48aba3$les588d73cpteefd92e8nmg6b3843$axsd29ee18
Decryptedtext:  plaintextmessage5d46f828bd89a39deb827ea4e31338c8
Matches original plaintext?:  True

Original text:  cryptographyrules
Plain_text:  cryptographyrulesf9be7d1af5bb9e35908497a2abd96b2d
Ciphertext:  pau91b97ddrgys7534abcoheefe826tplba90a9$yrrfdb59b2
Decryptedtext:  cryptographyrulesf9be7d1af5bb9e35908497a2ab

In [3]:
from itertools import permutations

def brute_force_attack(ciphertexts, alphabet=''.join(sorted(key)), max_key_length=9):
    for length in range(1, max_key_length + 1):
        for key_tuple in permutations(alphabet, length):
            key = ''.join(key_tuple)
            if all(check_key_with_ciphertext(key, ct) for ct in ciphertexts):
                return key
    return None  

def check_key_with_ciphertext(key, ciphertext):
    decrypted_text = decrypt_transposition(ciphertext, key)
    return pi_property(decrypted_text)

def run_brute_force():
    messages = [
        "helloalgorithm",
        "securetransmission",
        "plaintextmessage",
        "cryptographyrules",
        "federatedlearning"
    ]

    ciphertexts = [encrypt_transposition(make_plain_text(msg), key) for msg in messages]
    discovered_key = brute_force_attack(ciphertexts)
    if discovered_key:
        print(f"Key found: {discovered_key}")
        for ct in ciphertexts:
            decrypted_text = decrypt_transposition(ct, discovered_key)
            print(f"Decrypted Text: {decrypted_text}")
    else:
        print("No valid key found.")

run_brute_force()


Key found: cbead
Decrypted Text: helloalgorithmbf166fa043b9fd1b4b3ed9271db67922
Decrypted Text: securetransmission23b2023154ac651c86b42282c0e2605d
Decrypted Text: plaintextmessage5d46f828bd89a39deb827ea4e31338c8
Decrypted Text: cryptographyrulesf9be7d1af5bb9e35908497a2abd96b2d
Decrypted Text: federatedlearninge9f950395073828bcdd1c508138ce9f5
